# MÓDULO 4
## Tema 1. Lectura y escritura de archivos de texto en Python

**Objetivos:**
- Leer y escribir ficheros de texto con seguridad usando context managers (`with`).
- Controlar encoding, saltos de línea y manejo de errores.
- Usar `pathlib` para rutas robustas y multiplataforma.
- Procesar logs y generar informes de salida con ejemplos realistas.

## Índice
- Conceptos clave (ruta, encoding, newline, modos de apertura)
- Leer texto: `read`, `readline`, `readlines` e iteración
- Escribir texto: `write`, `writelines`, append y atomicidad
- Crear y procesar un log realista (filtrado, métricas básicas)
- Trampas comunes y buenas prácticas

## Conceptos clave

Cuando trabajas con ficheros de texto, las tres fuentes principales de problemas suelen ser:

- **Rutas** (relativas vs absolutas, separadores, permisos)
- **Encoding** (UTF-8, latin-1, caracteres raros)
- **Saltos de línea** (`\n` vs `\r\n`), especialmente al mover código entre sistemas

En Python, lo recomendable es:

- `pathlib.Path` para rutas
- `with open(...) as f:` para cerrar siempre
- `encoding="utf-8"` salvo que tengas un motivo para otra cosa

In [3]:
from pathlib import Path

ruta = Path(".") / "ejemplo.txt"
print("Ruta relativa:", ruta)
print("Ruta absoluta:", ruta.resolve())

Ruta relativa: ejemplo.txt
Ruta absoluta: F:\Compartida\MakeProjects\Formación\workspace\Tokio_CEMP\Python 2.0\Modulo_4\ejemplo.txt


In [4]:
ruta = Path("./datos") / "ejemplo.txt"
print("Ruta relativa:", ruta)
print("Ruta absoluta:", ruta.resolve())

Ruta relativa: datos\ejemplo.txt
Ruta absoluta: F:\Compartida\MakeProjects\Formación\workspace\Tokio_CEMP\Python 2.0\Modulo_4\datos\ejemplo.txt


In [5]:
try:
    BASE_DIR = Path(__file__).resolve().parent # Estamos en un script (.py)
except NameError:
    BASE_DIR = Path.cwd() # Estamos en un notebook (.ipynb)

print("BASE_DIR:", BASE_DIR)

BASE_DIR: f:\Compartida\MakeProjects\Formación\workspace\Tokio_CEMP\Python 2.0\Modulo_4


In [6]:
BASE_DIR = Path.cwd() # Estamos en un notebook (.ipynb)
year, month = 2024, 6
ruta = BASE_DIR / "datos" / str(year) / str(month) / "ejemplo.txt"
print("Ruta completa:", ruta)

Ruta completa: f:\Compartida\MakeProjects\Formación\workspace\Tokio_CEMP\Python 2.0\Modulo_4\datos\2024\6\ejemplo.txt


## Leer texto: `read`, `readline`, `readlines` e iteración

Formas típicas:

- `f.read()` → todo (ojo: tamaño)
- `f.readline()` → una línea
- `f.readlines()` → lista de líneas (ojo: tamaño)
- Iterar `for line in f:` → la opción más eficiente/idiomática para texto grande

In [ ]:
from pathlib import Path

# Creamos un fichero de ejemplo (solo para demo)
p = Path("demo_texto.txt")
p.write_text("INFO: Inicio\nWARNING: Algo raro\nERROR: Fallo crítico\n", encoding="utf-8")

# read()
with p.open("r", encoding="utf-8") as f:
    print("read():")
    print(f.read())

# readline()
with p.open("r", encoding="utf-8") as f:
    print("readline():")
    print(f.readline().strip())
    print(f.readline().strip())

# readlines()
with p.open("r", encoding="utf-8") as f:
    print("\nreadlines():")
    for line in f.readlines():
        print("\t->", line.strip())

# Iteración (forma idiomática)
with p.open("r", encoding="utf-8") as f:
    print("\nIteración línea a línea:")
    for line in f:
        print("->", line.strip())

with p.open("r", encoding="utf-8") as f:
    print("\nBuscando unicamente errores:")
    for line in f:
        if "ERROR" in line:
            print("->", line.strip())

read():
INFO: Inicio
ERROR: Fallo crítico

readline():
INFO: Inicio

readlines():
-> INFO: Inicio
-> WARNING: Algo raro
-> ERROR: Fallo crítico

Iteración línea a línea:
-> INFO: Inicio
-> WARNING: Algo raro
-> ERROR: Fallo crítico

Buscando unicamente errores:
-> ERROR: Fallo crítico


## Escribir texto: `write`, `writelines` y append

- Modo `"w"`: crea o sobrescribe el fichero.
- Modo `"a"`: añade contenido al final del fichero.
- `write()` escribe un string.
- `writelines()` escribe una secuencia de strings (no añade saltos de línea automáticamente).

In [ ]:
from pathlib import Path

p = Path("demo_escritura.txt")

# write() con modo "w" (sobrescribe)
with p.open("w", encoding="utf-8") as f:
    f.write("Primera línea\n")
    f.write("Segunda línea\n")

# writelines() con modo "w"
with p.open("w", encoding="utf-8") as f:
    f.writelines([
        "Línea A\n",
        "Línea B\n",
        "Línea C\n",
    ])

# append con modo "a" (añade al final)
with p.open("a", encoding="utf-8") as f:
    f.write("Línea añadida al final\n")

## Crear y procesar un log realista (filtrado y métricas)

Caso habitual en sistemas y aplicaciones: trabajar con un fichero de log
con distintos niveles de severidad:

- `INFO: ...` → información general
- `WARNING: ...` → situaciones anómalas no críticas
- `ERROR: ...` → fallos que requieren atención

A partir de este tipo de logs suele interesar:

- contar cuántos eventos hay de cada tipo
- identificar rápidamente los errores
- extraer las líneas `ERROR` para analizarlas o guardarlas aparte

Este patrón aparece constantemente en:
- backend y microservicios
- pipelines de datos
- monitorización y observabilidad
- scripts de mantenimiento


In [14]:
from pathlib import Path

# Crear un log realista
log_path = Path("app.log")
log_path.write_text(
    "INFO: Aplicación iniciada\n"
    "INFO: Conexión a base de datos\n"
    "WARNING: Tiempo de respuesta alto\n"
    "INFO: Petición recibida\n"
    "ERROR: No se pudo conectar al servicio externo\n"
    "INFO: Reintentando conexión\n"
    "ERROR: Timeout en la petición\n",
    encoding="utf-8"
)

# Procesar el log: conteo y extracción de errores
conteo = {"INFO": 0, "WARNING": 0, "ERROR": 0}
errores = []

with log_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line.startswith("INFO"):
            conteo["INFO"] += 1
        elif line.startswith("WARNING"):
            conteo["WARNING"] += 1
        elif line.startswith("ERROR"):
            conteo["ERROR"] += 1
            errores.append(line)

print("Conteo de eventos:")
for nivel, n in conteo.items():
    print(f"{nivel}: {n}")

print("\nLíneas de error:")
for e in errores:
    print("->", e)


Conteo de eventos:
INFO: 4
ERROR: 2

Líneas de error:
-> ERROR: No se pudo conectar al servicio externo
-> ERROR: Timeout en la petición


## Trampas comunes y buenas prácticas

- Usa `with open(...):` para no dejar ficheros abiertos.
- Define `encoding="utf-8"` explícitamente (evita sorpresas).
- No uses `read()`/`readlines()` en ficheros grandes sin pensarlo.
- Si el fichero puede contener caracteres “sucios”, prueba `errors="replace"` o `errors="ignore"` (pero documenta el impacto).
- Prefiere `pathlib.Path` a concatenar strings de rutas.

## Mini-práctica

1) Crea un fichero `notas.txt` con 10 líneas (puedes generarlo con Python).  
2) Léelo y cuenta cuántas líneas contienen la palabra `"Python"` (case-insensitive).  
3) Genera un reporte `resumen_notas.txt` con el número total de líneas y el número de coincidencias.